In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer

In [2]:
train = pd.read_csv('dataset/train.csv', index_col='Index')
train.head()

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
Index,,,,,,,,,,
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [3]:
df = train.copy()

# Remove duplicate rows
df = df.drop_duplicates()

# Standardize string columns
cat_cols = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

In [4]:
df['temp_missing'] = df['Temperature'].isna().astype(int)

df['weather_missing'] = (
    df['Weather'].isna()
).astype(int)

df['roadtype_missing'] = (
    df['RoadType'].isna()
).astype(int)

In [5]:
temp_median = df['Temperature'].median()

df['Temperature'] = (
    df['Temperature']
    .fillna(temp_median)
)

In [6]:
def parse_time(t):
    hour, minute = map(int, t.split(':'))
    return hour, minute

df[['hour','minute']] = (
    df['timestamp']
    .apply(lambda x: pd.Series(parse_time(x)))
)

In [7]:
df['total_minutes'] = (
    df['hour']*60
    + df['minute']
)

In [8]:
df['sin_time'] = np.sin(
    2*np.pi*df['total_minutes']/1440
)

df['cos_time'] = np.cos(
    2*np.pi*df['total_minutes']/1440
)

In [9]:
def get_time_bucket(hour):

    if 0 <= hour < 6:
        return 'Night'

    elif 6 <= hour < 12:
        return 'Morning'

    elif 12 <= hour < 18:
        return 'Afternoon'

    return 'Evening'

df['time_bucket'] = (
    df['hour']
    .apply(get_time_bucket)
)

In [10]:
def demand_period(hour):

    if hour <= 4:
        return "Night"

    elif hour <= 9:
        return "MorningRise"

    elif hour <= 14:
        return "Peak"

    elif hour <= 17:
        return "Decline"

    elif hour <= 20:
        return "LowDemand"

    return "Recovery"

df['demand_period'] = df['hour'].apply(demand_period)

In [11]:
temp_bins = [
    -np.inf,
    10,
    20,
    30,
    40,
    np.inf
]

temp_labels = [
    'VeryCold',
    'Cold',
    'Moderate',
    'Warm',
    'Hot'
]

df['temp_bin'] = pd.cut(
    df['Temperature'],
    bins=temp_bins,
    labels=temp_labels
)

In [12]:
df['temp_sq'] = (
    df['Temperature'] ** 2
)

In [13]:
df['weather_temp'] = (
    df['Weather'].astype(str)
    + '_'
    + df['temp_bin'].astype(str)
)

In [14]:
df['Landmarks_bin'] = (
    df['Landmarks']
    .map({
        'Yes':1,
        'No':0
    })
)

df['LargeVehicles_bin'] = (
    df['LargeVehicles']
    .map({
        'Allowed':1,
        'Not Allowed':0
    })
)

In [15]:
road_map = {
    'Highway':3,
    'Street':2,
    'Residential':1
}

df['road_score'] = (
    df['RoadType']
    .map(road_map)
)

In [16]:
df['road_capacity'] = (
    df['road_score']
    *
    df['NumberofLanes']
)

In [17]:
df['lane_vehicle_interaction'] = (
    df['NumberofLanes']
    *
    df['LargeVehicles_bin']
)

In [18]:
geohash_mean = (
    df.groupby('geohash')['demand']
    .mean()
)

df['geohash_mean_demand'] = (
    df['geohash']
    .map(geohash_mean)
)

In [19]:
weather_mean = (
    df.groupby('Weather')['demand']
    .mean()
)

df['weather_mean_demand'] = (
    df['Weather']
    .map(weather_mean)
)

In [20]:
geohash_hour_mean = (
    df.groupby(
        ['geohash','hour']
    )['demand']
    .mean()
)

df['geohash_hour_mean'] = (
    df.set_index(
        ['geohash','hour']
    ).index.map(
        geohash_hour_mean
    )
)

In [21]:
geohash_weather_mean = (
    df.groupby(
        ['geohash','Weather']
    )['demand']
    .mean()
)

df['geohash_weather_mean'] = (
    df.set_index(
        ['geohash','Weather']
    ).index.map(
        geohash_weather_mean
    )
)

In [22]:
df['sin_day'] = np.sin(
    2*np.pi*df['day']/7
)

df['cos_day'] = np.cos(
    2*np.pi*df['day']/7
)

In [23]:
df.head()

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,LargeVehicles_bin,road_score,road_capacity,lane_vehicle_interaction,geohash_mean_demand,weather_mean_demand,geohash_hour_mean,geohash_weather_mean,sin_day,cos_day
Index,,,,,,,,,,,,,,,,,,,,,
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,16.382587,NaN,...,0,NaN,NaN,0,0.040048,NaN,0.034855,NaN,-0.781831,0.62349
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,...,1,1.0,3.0,3,0.208766,0.094247,0.140998,0.211319,-0.781831,0.62349
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,...,0,1.0,1.0,0,0.127931,0.094247,0.068046,0.135150,-0.781831,0.62349
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,16.382587,Rainy,...,0,1.0,1.0,0,0.014381,0.094471,0.011243,0.011382,-0.781831,0.62349
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,...,0,1.0,1.0,0,0.029300,0.094471,0.020746,0.021256,-0.781831,0.62349


In [24]:
df.columns

Index(['geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'temp_missing',
       'weather_missing', 'roadtype_missing', 'hour', 'minute',
       'total_minutes', 'sin_time', 'cos_time', 'time_bucket', 'demand_period',
       'temp_bin', 'temp_sq', 'weather_temp', 'Landmarks_bin',
       'LargeVehicles_bin', 'road_score', 'road_capacity',
       'lane_vehicle_interaction', 'geohash_mean_demand',
       'weather_mean_demand', 'geohash_hour_mean', 'geohash_weather_mean',
       'sin_day', 'cos_day'],
      dtype='str')

In [25]:
df.to_csv('dataset/feature_engineered_train.csv', index=False)